In [1]:
import xgboost as xgb

print(xgb.__version__)

3.4.1


In [ ]:
from sklearn import set_config

# Jupyter에서 sklearn 모델을 HTML diagram 대신 text로 표시
set_config(display="text")

In [2]:
from xgboost import XGBClassifier

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Raw Data에서 다시 시작
df = pd.read_csv("../01_raw/uci-secom.csv")

# 공정 측정 Feature 0 ~ 589
sensor_cols = [str(i) for i in range(590)]

# X / y 정의
X = df[sensor_cols].copy()

# Pass = 0, Fail = 1
y = (df["Pass/Fail"] == 1).astype(int)

# Phase 3과 동일한 Train/Test Split 재현
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

print("\nTrain Target:")
print(y_train.value_counts())

print("\nTest Target:")
print(y_test.value_counts())

Train Shape: (1253, 590)
Test Shape : (314, 590)

Train Target:
Pass/Fail
0    1170
1      83
Name: count, dtype: int64

Test Target:
Pass/Fail
0    293
1     21
Name: count, dtype: int64


In [6]:
baseline_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

baseline_xgb.fit(
    X_train,
    y_train
);

In [7]:
print("Baseline XGBoost training completed.")

Baseline XGBoost training completed.


In [8]:
# 최종 예측 클래스
y_pred = baseline_xgb.predict(X_test)

# Fail(1)일 확률
y_prob = baseline_xgb.predict_proba(X_test)[:, 1]

In [11]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
ap = average_precision_score(y_test, y_prob)

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix")
print(cm)

print()
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"Average Precision (AP)   : {ap:.4f}")

Confusion Matrix
[[292   1]
 [ 21   0]]

Precision: 0.0000
Recall   : 0.0000
F1-score : 0.0000
Average Precision (AP)   : 0.1674


In [10]:
# Precision: 모델이 " 이 샘플 불량 같습니다." 라고 알람을 띄운 것 중 실제 불량의 비율
# Recall : 실제 불량 중 모델이 몇 %를 잡아냈는가
# 다만, 그렇다고 Recall이 모두 100이 최고는 아니다. 왜냐하면, 모든 제품을 모두 Fail이라고 찍으면 Recall은 100이 되지만 정상 제품까지 전부 Fail로 잡아버리는 오류가 있기 때문
#F-1 score : 불량을 많이 잡음녀서고, 쓸데없는 불량 경고를 남발하지 않는가?

A0 — Native NaN + Unweighted XGBoost

TN = 292
FP = 1
FN = 21
TP = 0

Precision = 0.000
Recall = 0.000
F1 = 0.000
AP = 0.167

→ Default model predicted almost all samples as Pass.
→ No Fail samples were detected at the default classification decision.
→ Class imbalance handling is required for the next experiment.

In [12]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

In [13]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [14]:
cv_results = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train, y_train),
    start=1
):
    # 현재 Fold의 학습/검증 데이터
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    # 현재 Fold의 학습 데이터에서만 class weight 계산
    negative_count = (y_fold_train == 0).sum()
    positive_count = (y_fold_train == 1).sum()

    scale_pos_weight = negative_count / positive_count

    # -------------------------
    # A0: 가중치 없는 baseline
    # -------------------------
    model_a0 = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    )

    # -------------------------
    # A1: class weight 적용
    # -------------------------
    model_a1 = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        scale_pos_weight=scale_pos_weight
    )

    models = {
        "A0_Unweighted": model_a0,
        "A1_Weighted": model_a1
    }

    for model_name, model in models.items():

        model.fit(
            X_fold_train,
            y_fold_train
        )

        y_val_pred = model.predict(X_fold_val)

        y_val_prob = model.predict_proba(
            X_fold_val
        )[:, 1]

        precision = precision_score(
            y_fold_val,
            y_val_pred,
            zero_division=0
        )

        recall = recall_score(
            y_fold_val,
            y_val_pred
        )

        f1 = f1_score(
            y_fold_val,
            y_val_pred
        )

        ap = average_precision_score(
            y_fold_val,
            y_val_prob
        )

        tn, fp, fn, tp = confusion_matrix(
            y_fold_val,
            y_val_pred
        ).ravel()

        cv_results.append({
            "Fold": fold,
            "Model": model_name,
            "Scale_Pos_Weight": (
                1.0
                if model_name == "A0_Unweighted"
                else scale_pos_weight
            ),
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "AP": ap
        })

In [15]:
cv_results_df = pd.DataFrame(cv_results)

display(
    cv_results_df.round(4)
)

,Fold,Model,Scale_Pos_Weight,TN,FP,FN,TP,Precision,Recall,F1,AP
0,1,A0_Unweighted,1.0000,233,1,17,0,0.0,0.0000,0.0000,0.1552
1,1,A1_Weighted,14.1818,232,2,17,0,0.0,0.0000,0.0000,0.0909
2,2,A0_Unweighted,1.0000,233,1,16,1,0.5,0.0588,0.1053,0.2641
3,2,A1_Weighted,14.1818,232,2,15,2,0.5,0.1176,0.1905,0.2199
4,3,A0_Unweighted,1.0000,234,0,17,0,0.0,0.0000,0.0000,0.1505
5,3,A1_Weighted,14.1818,233,1,17,0,0.0,0.0000,0.0000,0.2558
6,4,A0_Unweighted,1.0000,232,2,16,0,0.0,0.0000,0.0000,0.0839
7,4,A1_Weighted,13.9701,234,0,16,0,0.0,0.0000,0.0000,0.1184
8,5,A0_Unweighted,1.0000,233,1,16,0,0.0,0.0000,0.0000,0.1505
9,5,A1_Weighted,13.9701,230,4,16,0,0.0,0.0000,0.0000,0.1332


In [16]:
cv_summary = (
    cv_results_df
    .groupby("Model")[
        ["Precision", "Recall", "F1", "AP"]
    ]
    .agg(["mean", "std"])
)

display(
    cv_summary.round(4)
)

Precision          Recall              F1              AP  \
                   mean     std    mean     std    mean     std    mean   
Model                                                                     
A0_Unweighted       0.1  0.2236  0.0118  0.0263  0.0211  0.0471  0.1609   
A1_Weighted         0.1  0.2236  0.0235  0.0526  0.0381  0.0852  0.1636   

                       
                  std  
Model                  
A0_Unweighted  0.0649  
A1_Weighted    0.0706

In [17]:
# Class Weight가 일부 샘플을 0.5 threshold 너머로 밀어 Fail 판정을 조금 더 만들기는 했지만, 모델의 전체적인 확률 ranking 능력을 크게 개선하지는 못했다.라는 가정이 합리적

In [18]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [19]:
model_b0 = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "xgb",
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42
        )
    )
])

In [20]:
b0_results = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train, y_train),
    start=1
):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    model_b0 = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "xgb",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42
            )
        )
    ])

    model_b0.fit(
        X_fold_train,
        y_fold_train
    )

    y_val_pred = model_b0.predict(
        X_fold_val
    )

    y_val_prob = model_b0.predict_proba(
        X_fold_val
    )[:, 1]

    precision = precision_score(
        y_fold_val,
        y_val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_fold_val,
        y_val_pred
    )

    f1 = f1_score(
        y_fold_val,
        y_val_pred
    )

    ap = average_precision_score(
        y_fold_val,
        y_val_prob
    )

    tn, fp, fn, tp = confusion_matrix(
        y_fold_val,
        y_val_pred
    ).ravel()

    b0_results.append({
        "Fold": fold,
        "Model": "B0_Median_Indicator",
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "AP": ap
    })

In [21]:
b0_results_df = pd.DataFrame(b0_results)

display(
    b0_results_df.round(4)
)

,Fold,Model,TN,FP,FN,TP,Precision,Recall,F1,AP
0,1,B0_Median_Indicator,232,2,16,1,0.3333,0.0588,0.1000,0.1847
1,2,B0_Median_Indicator,233,1,16,1,0.5000,0.0588,0.1053,0.2887
2,3,B0_Median_Indicator,233,1,17,0,0.0000,0.0000,0.0000,0.1582
3,4,B0_Median_Indicator,234,0,16,0,0.0000,0.0000,0.0000,0.1294
4,5,B0_Median_Indicator,234,0,16,0,0.0000,0.0000,0.0000,0.1558


In [22]:
b0_summary = (
    b0_results_df[
        ["Precision", "Recall", "F1", "AP"]
    ]
    .agg(["mean", "std"])
)

display(
    b0_summary.round(4)
)

,Precision,Recall,F1,AP
mean,0.1667,0.0235,0.0411,0.1833
std,0.2357,0.0322,0.0562,0.0621


In [23]:
a0_results = cv_results_df[
    cv_results_df["Model"] == "A0_Unweighted"
][
    [
        "Fold",
        "Model",
        "TN",
        "FP",
        "FN",
        "TP",
        "Precision",
        "Recall",
        "F1",
        "AP"
    ]
].copy()

missing_strategy_compare = pd.concat(
    [
        a0_results,
        b0_results_df
    ],
    ignore_index=True
)

missing_strategy_summary = (
    missing_strategy_compare
    .groupby("Model")[
        ["Precision", "Recall", "F1", "AP"]
    ]
    .agg(["mean", "std"])
)

display(
    missing_strategy_summary.round(4)
)

Precision          Recall              F1              AP  \
                         mean     std    mean     std    mean     std    mean   
Model                                                                           
A0_Unweighted          0.1000  0.2236  0.0118  0.0263  0.0211  0.0471  0.1609   
B0_Median_Indicator    0.1667  0.2357  0.0235  0.0322  0.0411  0.0562  0.1833   

                             
                        std  
Model                        
A0_Unweighted        0.0649  
B0_Median_Indicator  0.0621

In [24]:
b1_results = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train, y_train),
    start=1
):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    # 현재 Fold의 Train 부분만 사용해서 class weight 계산
    negative_count = (y_fold_train == 0).sum()
    positive_count = (y_fold_train == 1).sum()

    scale_pos_weight = negative_count / positive_count

    model_b1 = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "xgb",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                scale_pos_weight=scale_pos_weight
            )
        )
    ])

    model_b1.fit(
        X_fold_train,
        y_fold_train
    )

    y_val_pred = model_b1.predict(X_fold_val)

    y_val_prob = model_b1.predict_proba(
        X_fold_val
    )[:, 1]

    precision = precision_score(
        y_fold_val,
        y_val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_fold_val,
        y_val_pred
    )

    f1 = f1_score(
        y_fold_val,
        y_val_pred
    )

    ap = average_precision_score(
        y_fold_val,
        y_val_prob
    )

    tn, fp, fn, tp = confusion_matrix(
        y_fold_val,
        y_val_pred
    ).ravel()

    b1_results.append({
        "Fold": fold,
        "Model": "B1_Median_Indicator_Weighted",
        "Scale_Pos_Weight": scale_pos_weight,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "AP": ap
    })

In [25]:
b1_results_df = pd.DataFrame(b1_results)

display(
    b1_results_df.round(4)
)

,Fold,Model,Scale_Pos_Weight,TN,FP,FN,TP,Precision,Recall,F1,AP
0,1,B1_Median_Indicator_Weighted,14.1818,232,2,15,2,0.50,0.1176,0.1905,0.1386
1,2,B1_Median_Indicator_Weighted,14.1818,231,3,16,1,0.25,0.0588,0.0952,0.2465
2,3,B1_Median_Indicator_Weighted,14.1818,233,1,17,0,0.00,0.0000,0.0000,0.2307
3,4,B1_Median_Indicator_Weighted,13.9701,234,0,16,0,0.00,0.0000,0.0000,0.1328
4,5,B1_Median_Indicator_Weighted,13.9701,233,1,16,0,0.00,0.0000,0.0000,0.1263


In [26]:
b1_summary = (
    b1_results_df[
        ["Precision", "Recall", "F1", "AP"]
    ]
    .agg(["mean", "std"])
)

display(
    b1_summary.round(4)
)

,Precision,Recall,F1,AP
mean,0.1500,0.0353,0.0571,0.1750
std,0.2236,0.0526,0.0852,0.0585


In [27]:
all_cv_results = pd.concat(
    [
        cv_results_df[
            cv_results_df["Model"].isin(
                ["A0_Unweighted", "A1_Weighted"]
            )
        ],
        b0_results_df,
        b1_results_df
    ],
    ignore_index=True
)

model_comparison = (
    all_cv_results
    .groupby("Model")[
        ["Precision", "Recall", "F1", "AP"]
    ]
    .agg(["mean", "std"])
)

display(
    model_comparison.round(4)
)

Precision          Recall              F1  \
                                  mean     std    mean     std    mean   
Model                                                                    
A0_Unweighted                   0.1000  0.2236  0.0118  0.0263  0.0211   
A1_Weighted                     0.1000  0.2236  0.0235  0.0526  0.0381   
B0_Median_Indicator             0.1667  0.2357  0.0235  0.0322  0.0411   
B1_Median_Indicator_Weighted    0.1500  0.2236  0.0353  0.0526  0.0571   

                                          AP          
                                 std    mean     std  
Model                                                 
A0_Unweighted                 0.0471  0.1609  0.0649  
A1_Weighted                   0.0852  0.1636  0.0706  
B0_Median_Indicator           0.0562  0.1833  0.0621  
B1_Median_Indicator_Weighted  0.0852  0.1750  0.0585